# Multi Model Evaluation

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/brightbandtech/extremeweatherbench/blob/main/notebooks/multi_model_evaluation.ipynb)

This notebook demonstrates how to compare multiple CIRA MLWP models against a shared target. Optimized for Google Colab with a small demo case.

In [ ]:
!pip install -q extremeweatherbench

In [ ]:
import datetime
import extremeweatherbench as ewb
from extremeweatherbench import inputs
from extremeweatherbench.cases import IndividualCase
from extremeweatherbench.regions import BoundingBoxRegion

# Mini-case: 2020 SW US Heat Wave — Colab-optimized
demo_case = IndividualCase(
    case_id_number=9010,
    title="2020 SW US Heat Wave (demo)",
    start_date=datetime.datetime(2020, 8, 15),
    end_date=datetime.datetime(2020, 8, 18),
    location=BoundingBoxRegion.create_region(
        latitude_min=34.0,
        latitude_max=40.0,
        longitude_min=242.0,
        longitude_max=248.0,
    ),
    event_type="heat_wave",
)
cases = [demo_case]

In [ ]:
model_names = [
    "FOUR_v200_IFS",
    "FOUR_v200_GFS",
    "GRAP_v100_IFS",
    "AURO_v100_IFS",
]

target = ewb.ERA5(
    variables=["surface_air_temperature"]
)

metrics_list = [
    ewb.metrics.MeanAbsoluteError(
        forecast_variable="surface_air_temperature",
        target_variable="surface_air_temperature",
    ),
    ewb.metrics.MaximumMeanAbsoluteError(
        forecast_variable="surface_air_temperature",
        target_variable="surface_air_temperature",
    ),
    ewb.metrics.RootMeanSquaredError(
        forecast_variable="surface_air_temperature",
        target_variable="surface_air_temperature",
    ),
]

eval_objects = [
    ewb.EvaluationObject(
        event_type="heat_wave",
        metric_list=metrics_list,
        target=target,
        forecast=inputs.get_cira_icechunk(
            model_name=name
        ),
    )
    for name in model_names
]

In [ ]:
runner = ewb.evaluation(
    case_metadata=cases,
    evaluation_objects=eval_objects,
)
outputs = runner.run_evaluation()

mae = outputs[outputs["metric"] == "MeanAbsoluteError"]
pivot = (
    mae.groupby(["forecast_source", "lead_time"])["value"]
    .mean()
    .unstack("forecast_source")
)
print(pivot)